# Module 8: Full Pipeline Orchestrator

This notebook runs the complete end-to-end multi-agent pipeline for Reliance Industries (BSE: 500325, NSE: RELIANCE, Q4FY26). It compiles the execution step timings, displays the top Q&A pairs, verifies outputs, and showcases a feedback stub to adjust topic weighting for future quarters.

In [ ]:
# Cell 1: Install verify and Imports
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from pipeline import EarningsCallPipeline
print("Pipeline module imported successfully.")

In [ ]:
# Cell 2: Instantiate EarningsCallPipeline
pipeline = EarningsCallPipeline()
print("EarningsCallPipeline instantiated.")

In [ ]:
# Cell 3: Run Pipeline for Reliance Industries
print("Running EarningsCallPipeline for Reliance Industries...")
result = await pipeline.run(
    company="Reliance Industries",
    bse_code="500325",
    nse_symbol="RELIANCE",
    quarter="Q4FY26"
)
print("Pipeline execution complete!")

In [ ]:
# Cell 4: Timing breakdown per pipeline step
import pandas as pd
from IPython.display import display, HTML

timings = result["timings"]
df_timings = pd.DataFrame(list(timings.items()), columns=["Pipeline Step", "Duration (seconds)"])
display(HTML("<h3>Step-by-Step Pipeline Timing Breakdown</h3>"))
display(df_timings)

In [ ]:
# Cell 5: Display Top 5 QAPairs
from IPython.display import display, Markdown

display(Markdown("## Top 5 Predicted Q&A Pairs"))
validated_results = result["validation_results"]

for idx, res in enumerate(validated_results[:5]):
    qa = res.qa_pair
    diff_pct = int(qa.adversarial_score * 100)
    md_output = f"""
**Q{idx+1} ({qa.topic}) — Difficulty: {diff_pct}%**
*Question*: {qa.question}

*Suggested CFO Response*:
    > {res.final_answer}

*Why it is tough*: {qa.why_tough}

--- (Validation: Passed={res.validation_passed}, Factuality={res.factual_score}, Tone={res.tone_score})
"""
    display(Markdown(md_output))

In [ ]:
# Cell 6: PPTX Slide Count + Cheat Sheet Word Count
from pptx import Presentation

cheat_sheet_path = result["outputs"]["cheat_sheet_path"]
pptx_path = result["outputs"]["pptx_path"]

with open(cheat_sheet_path, "r", encoding="utf-8") as f:
    words = len(f.read().split())
    
prs = Presentation(pptx_path)
slides_count = len(prs.slides)

print("=== GENERATED PACKAGE VERIFICATION ===")
print(f"Briefing Cheat Sheet File: {cheat_sheet_path}")
print(f"Briefing Cheat Sheet Word Count: {words} words")
print(f"Slide Deck Presentation File: {pptx_path}")
print(f"Slide Deck Presentation Slide Count: {slides_count} slides")

In [ ]:
# Cell 7: Post-Call Feedback Loop Stub
print("=== POST-CALL FEEDBACK LOOP STUB ===")

def apply_post_call_feedback(company: str, actual_asked_topics: list[str]):
    """
    Simulates post-call feedback from investor relations, increasing the recurrence/weight 
    of the topics that analysts actually asked about during the call inside the cold_store metadata.
    """
    print(f"Applying feedback for {company}...")
    # Access cold store collection
    cold_col = pipeline.extraction_agent.cold_collection
    
    # Fetch chunks for this company
    results = cold_col.get(where={"company": company})
    ids = results["ids"]
    metadatas = results["metadatas"]
    
    updated_count = 0
    for idx, (cid, meta) in enumerate(zip(ids, metadatas)):
        topic = meta.get("topic_tag", "")
        # If the chunk's topic matches one of the actually asked topics, boost its importance
        # or mark it with a feedback flag
        if topic in actual_asked_topics:
            meta["feedback_boost"] = True
            meta["sentiment"] -= 0.1  # assume increased concern/negativity if asked about
            cold_col.update(ids=[cid], metadatas=[meta])
            updated_count += 1
            
    print(f"Boosted {updated_count} historical analyst turns associated with topics: {actual_asked_topics}")

# Simulate that analysts actually asked about 'Margins' and 'Capex' during the call
apply_post_call_feedback("Reliance Industries", ["Margins", "Capex"])